In [3]:
import os
import re
import warnings
import random
from collections import defaultdict
from typing import Dict, List, Tuple

import torch
import torch.nn.functional as F
import numpy as np
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings("ignore")

In [4]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

## Задание

1) Реализовать методы `greedy_sampling` и `generate` (1 балл)
2) Реализовать метод `random_sampling` и поддержать его в `generate` (1 балл)
3) Реализовать метод `_beam_search_generate` и поддержать его в `generate` (2 балла)
4) Реализовать методы `apply_top_p`, `apply_top_k`, `apply_temperature` и поддержать их в `generate` (1 балл)  
Все методы необходимо реализовать через векторные операции в torch/numpy везде где это возможно

In [52]:
class Model:
    def __init__(self, model_name: str = "gpt2"):
        self.model = GPT2LMHeadModel.from_pretrained(model_name)
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.vocab_size = self.tokenizer.vocab_size

        self.valid_strategies = {"greedy", "random_sampling", "beam_search"}

    def greedy_sampling(self, logits: torch.Tensor) -> torch.Tensor:
        return torch.argmax(logits, dim=-1)

    def random_sampling(self, logits: torch.Tensor, temperature: float, top_k: int, top_p: int) -> torch.Tensor:
        logits_t = self._apply_temperature(logits, temperature)
        logits_t_top_k = self._apply_top_k(logits_t, top_k)
        final_logits = self._apply_top_p(logits_t_top_k, top_p)

        probs = torch.softmax(final_logits, dim=-1)

        return torch.multinomial(probs, num_samples=1)[0]

    def _beam_search_generate(
        self,
        prompt: str,
        max_length: int = 50,
        num_beams: int = 3,
        length_penalty: float = 1.0,
        early_stopping: bool = True
    ) -> str:
        input_ids = self.tokenizer(prompt, return_tensors="pt")["input_ids"]
        device = next(self.model.parameters()).device
        input_ids = input_ids.to(device)
        vocab_size = self.vocab_size

        cur_len = input_ids.shape[1]
        max_len = min(cur_len + max_length, 1024)

        input_ids = input_ids.repeat(num_beams, 1)
        beam_scores = torch.zeros(num_beams, dtype=torch.float, device=device)

        done = [False] * num_beams
        generated_hyps = []  # список завершённых гипотез

        for step in range(cur_len, max_len):
            with torch.no_grad():
                outputs = self.model(input_ids)
                next_token_logits = outputs.logits[:, -1, :]  # [num_beams, vocab_size]
                next_token_logprobs = torch.log_softmax(next_token_logits, dim=-1)

            next_scores = beam_scores.unsqueeze(-1) + next_token_logprobs  # [num_beams, vocab_size]
            next_scores = next_scores.view(-1)  # [num_beams * vocab_size]

            # Берём top (num_beams * 2) кандидатов
            topk_scores, topk_indices = torch.topk(next_scores, num_beams * 2, dim=0)
            beam_indices = topk_indices // vocab_size
            token_indices = topk_indices % vocab_size

            next_beam_scores = []
            next_input_ids = []
            next_done = []

            for score, beam_idx, token_id in zip(topk_scores, beam_indices, token_indices):
                beam_idx = beam_idx.item()
                token_id = token_id.item()
                beam_score = score.item()

                if done[beam_idx]:
                    continue

                new_input = torch.cat(
                    [input_ids[beam_idx], torch.tensor([token_id], device=device)]
                )

                is_eos = (token_id == self.tokenizer.eos_token_id)

                # Добавляем гипотезу, если beam завершился (сгенерировался eos)
                if is_eos:
                    norm_score = beam_score / (len(new_input) ** length_penalty)
                    generated_hyps.append({"ids": new_input, "score": norm_score})
                else:
                    next_input_ids.append(new_input)
                    next_beam_scores.append(beam_score)
                    next_done.append(False)


                if len(next_input_ids) >= num_beams:
                    break

            if len(next_input_ids) == 0:
                break

            input_ids = torch.nn.utils.rnn.pad_sequence(
                next_input_ids, batch_first=True, padding_value=self.tokenizer.eos_token_id
            )
            beam_scores = torch.tensor(next_beam_scores, device=device)
            done = next_done

            if early_stopping and len(generated_hyps) >= num_beams:
                break

        # Добавляем незавершённые beam'ы
        for i in range(len(input_ids)):
            if not done[i]:
                norm_score = beam_scores[i].item() / (input_ids[i].shape[0] ** length_penalty)
                generated_hyps.append({"ids": input_ids[i], "score": norm_score})

        # Выбор лучшей гипотезы
        if generated_hyps:
            best_hyp = max(generated_hyps, key=lambda x: x["score"])
            best_hyp_ids = best_hyp["ids"]
        else:
            best_idx = beam_scores.argmax()
            best_hyp_ids = input_ids[best_idx]

        prompt_len = len(self.tokenizer(prompt)["input_ids"])
        output_ids = best_hyp_ids[prompt_len:]

        return self.tokenizer.decode(output_ids, skip_special_tokens=True)


    def _apply_temperature(self, logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        return logits / temperature

    def _apply_top_p(self, logits: torch.Tensor, top_p: float = 1.0) -> torch.Tensor:
        if top_p == 1.0:
            return logits

        probs = torch.softmax(logits, dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        sorted_indices_to_remove = cumulative_probs > top_p
        # Сдвигаем на 1 вправо, чтобы включить последний токен, не превысивший p
        # Например, cumulative_probs - [0.7, 0.9, 0.95, 1], top_p = 0.85, тогда 0.7 не хватает до 0.85, нужен еще один токен
        sorted_indices_to_remove = torch.cat(
            [torch.zeros_like(sorted_indices_to_remove[:, :1]), sorted_indices_to_remove[:, :-1]],
            dim=-1
        )

        indices_to_remove = torch.zeros_like(logits, dtype=torch.bool)
        indices_to_remove.scatter_(dim=-1, index=sorted_indices, src=sorted_indices_to_remove)

        filtered_logits = logits.clone()
        filtered_logits[indices_to_remove] = -float('inf')

        return filtered_logits

    def _apply_top_k(self, logits: torch.Tensor, top_k: int = 0) -> torch.Tensor:
        if top_k == 0 or top_k == self.vocab_size:
            filtered_logits = logits
        else:
            filtered_logits = logits.clone()
            topk_vals, _ = torch.topk(filtered_logits, top_k)
            kth_val = topk_vals[:, -1]
            filtered_logits[filtered_logits < kth_val] = -float('inf') # Это даст нулевые значения вероятностей для соответствующих элементов в softmax

        return filtered_logits

    def generate(
        self,
        prompt: str,
        max_length: int = 50,
        strategy: str = "greedy",
        temperature: float = 1.0,
        top_k: int = 0,
        top_p: float = 1.0,
        num_beams: int = 3
    ) -> str:
        if temperature < 0:
            raise ValueError(f"Недопустимое значение аргумента temperature")

        if top_p > 1.0 or top_p < 0.0:
            raise ValueError(f"Недопустимое значение аргумента top_p")

        if top_k < 0:
            raise ValueError(f"Недопустимое значение аргумента top_k")

        if strategy not in self.valid_strategies:
            raise ValueError(
                f"Недопустимое значение аргумента strategy: {strategy}. "
                f"Допустимые значения: {sorted(self.valid_strategies)}"
            )

        if max_length <= 0:
            raise ValueError(f"Недопустимое значение аргумента max_length")

        if num_beams <= 0:
            raise ValueError(f"Недопустимое значение агрумента num_beams")

        if top_k > self.vocab_size:
            warnings.warn(
                "Значение top_k превышает размер словаря. "
                "Семплирование будет производиться из всех возможных токенов!",
                Warning
            )

        if num_beams > self.vocab_size:
            warnings.warn(
                "Значение num_beams превышает размер словаря. "
                "Поиск будет производиться по всем возможным комбинациям токенов!",
                Warning
            )

        if strategy == "beam_search":
             return self._beam_search_generate(prompt, max_length, num_beams)

        tokens = self.tokenizer(prompt, return_tensors="pt")
        prompt_length = tokens["input_ids"].size(1)

        for i in range(max_length):
            with torch.no_grad():
                outputs = self.model(**tokens)
                logits = outputs.logits[:, -1, :] # Для последнего токена

                if strategy == "greedy" or strategy == "random_sampling" and temperature < 1e-6:
                    next_token = self.greedy_sampling(logits)
                elif strategy == "random_sampling":
                    next_token = self.random_sampling(logits, temperature, top_k, top_p)

                if next_token.item() == self.tokenizer.eos_token:
                    break

                tokens = {
                    "input_ids": torch.cat([tokens["input_ids"], next_token.unsqueeze(0)], dim=1),
                    "attention_mask": torch.cat([tokens["attention_mask"], torch.tensor([[1]])], dim=1)
                }

        output_tokens = tokens["input_ids"][0, prompt_length:]

        output = self.tokenizer.decode(output_tokens)
        return output

Протестируем реализованные алгоритмы:

In [53]:
model = Model()

### Жадный алгоритм

In [106]:
model.generate("What color is the sky?", strategy='greedy')

"\n\nThe sky is the most beautiful thing in the world. It's the most beautiful thing in the world because it's the only thing that can change the world. It's the only thing that can change the world because it's the only thing"

### Случайное семплирование

При temperature = 0 под случайным семплирование понимется жадный подход:

In [108]:
model.generate("What color is the sky?", strategy="random_sampling", temperature=0.0)

"\n\nThe sky is the most beautiful thing in the world. It's the most beautiful thing in the world because it's the only thing that can change the world. It's the only thing that can change the world because it's the only thing"

Значение температуры по умолчанию = 1.0

In [107]:
model.generate("What color is the sky?", strategy="random_sampling")

" When we think of PLEX/EMP, we think of an artificial world in space and time.\n\n\nFRIDAY, 18th April 2015:: So of course, Avacyn is part of APM's reality mode. People are"

Генерация с большой вероятностью будет не связана в запросом. Попробуем уменьшить температуру:

In [109]:
model.generate("What color is the sky?", strategy="random_sampling", temperature=0.5)

'\n\nThe answer is blue.\n\nThe sky is blue.\n\nThe sky is blue.\n\nThe sky is blue.\n\nThe sky is blue.\n\nThe sky is blue.\n\nThe sky is blue.\n'

Получился даже правильный ответ

#### Top K

При top_k = 1 ожидается резульатат, как при жадном подходе:

In [127]:
model.generate("What color is the sky?", strategy="random_sampling", top_k=1)

"\n\nThe sky is the most beautiful thing in the world. It's the most beautiful thing in the world because it's the only thing that can change the world. It's the only thing that can change the world because it's the only thing"

При большем значении top_k генерация должна быть еще по теме запроса, но отличаться от жадного подхода

In [128]:
model.generate("What color is the sky?", strategy="random_sampling", top_k=2)

'\n\nThe sky is a color that is not always clear to the eye. It is a combination of light and dark. The sky is a combination of light and dark, which is the color that is most often seen when the sun is in the'

При большом значении top_k ожидается, что генерация будет не релевантной

In [129]:
model.generate("What color is the sky?", strategy="random_sampling", top_k=100)

'\n\nDance in\n\nPraise on the Moon for the "\n\nby\n\nMarjorie Schue in Moonlight\'s Place\n\n"\n\nFor the history of a race, find the moon with ", even though'

#### Top p

Возьмем предыдущий пример и зададим также небольшое значение top_p. Это позволит сильно ограничить диапазон для семплирования:

In [31]:
model.generate("What color is the sky?", strategy="random_sampling", top_k=100, top_p=0.4)

"\n\nThe sky is a bright spot.\n\nIt's the only thing that makes it bright.\n\nThe sky is the only thing that makes it bright.\n\nThe sky is the only thing that makes it bright.\n\nThe"

При top_p = 0.0 ожидается результат, как при жадном подходе:

In [147]:
model.generate("What color is the sky?", strategy="random_sampling", top_k=100, top_p=0.)

"\n\nThe sky is the most beautiful thing in the world. It's the most beautiful thing in the world because it's the only thing that can change the world. It's the only thing that can change the world because it's the only thing"

### Beam search

При num_beams = 1 также ожидается результат, как при жадном подходе:

In [9]:
model.generate("What color is the sky?", strategy="beam_search", num_beams=1)

"\n\nThe sky is the most beautiful thing in the world. It's the most beautiful thing in the world because it's the only thing that can change the world. It's the only thing that can change the world because it's the only thing"

При большем значении num_beams результат может отличаться от жадного подхода. Однако жадный подход может дать решение с лучшим или около оптимальным правдоподобием. Поэтому beam_seacrh может не получить другой результат, либо получить его при довольно большом занчении num_beams:

In [10]:
model.generate("What color is the sky?", strategy="beam_search", num_beams=5)

"\n\nThe sky is the most beautiful thing in the world. It's the most beautiful thing in the world because it's the only thing that can change the world. It's the only thing that can change the world because it's the only thing"

Еще пример:

In [166]:
model.generate("The only thing we have to fear is", strategy='greedy')

' that the government will not be able to stop us from doing what we want to do," he said.\n\n"We have to be prepared to do what we want to do. We have to be prepared to do what we want to do.'

In [168]:
model.generate("The only thing we have to fear is", strategy="beam_search", num_beams=100)

' that the government will not be able to stop us from doing what we want to do," he said.\n\n"We have to be prepared to do what we want to do. We have to be prepared to do what we want to do.'